# GR00T-N1.7 fine-tune on any `robotdataset` dataset

Standalone notebook: no `launch_finetune.py` subprocess, no sbatch wrapper. The model and its processor
are GR00T's own classes, used directly instead of through the `Gr00tN1d7Pipeline` / HF-`Trainer` machinery.

**The dataset is only ever touched through `robotdataset`**, in two layers:

1. **Dataset info** (`robotdataset.info`): the loader lists every modality it provides (path, kind, dtype,
   shape) as an editable YAML. You choose which ones feed the model (`use: true`), their order, their
   model-side names, how a vector leaf is cut into groups (`split`) and how each action group is represented.
2. **Model adapter** (`robotdataset.adapters.gr00t.Gr00tAdapter`): reads that YAML and builds the loader with
   the right sampling windows, the GR00T `ModalityConfig`s, the `VLAStepData` for every batch and the
   normalisation statistics the processor needs. The statistics are computed by the loader over the
   training episodes, in memory; nothing reads `meta/*.json`.

Two files describe a run, and neither repeats the other's content:

| file | holds | read with |
|---|---|---|
| run file, `FINETUNE_CONFIG` (default `configs/scratch/stack_tact_v4.yaml`) | model, training, `data.train_episodes`, `data.val_episodes`, `data.modality_file` | `training_cfgs.Config` |
| modality file (dataset-info YAML) | loader and its arguments, selected and ordered modalities, action representation, windows | `Gr00tAdapter.from_yaml` |

Every hyperparameter, path and split comes from those two files; none is hardcoded here. Episode lists are
ints or `"a:b"` half-open ranges. To train on another dataset, list it once, edit the file, and point
`data.modality_file` at it:

```bash
python -m robotdataset.info --loader OXEDataset --kwargs '{"dataset_name": "viola"}' \
    --episodes 0:8 --adapter gr00t -o configs/scratch/my_dataset.modality.yaml
```

Run this inside `$WORK/venv/finetune-groot` (`module load FFmpeg/7.1.2` first).

In [ ]:
import os
import shutil
import sys
from pathlib import Path

import numpy as np
import torch
import wandb
from training_cfgs import Config

try:  # OXEDataset reads TFDS through TensorFlow; keep it off the GPU so it cannot take memory from torch
    import tensorflow as tf

    tf.config.set_visible_devices([], "GPU")
except ImportError:
    pass

REPO_ROOT = Path(os.environ.get("FINETUNE_REPO", Path.cwd())) #/ "finetune-groot"))
sys.path.insert(0, str(REPO_ROOT / "vendor" / "Isaac-GR00T"))

# Single source of truth for hyperparameters, paths and the train/val split.
RUN_CONFIG = Path(os.environ.get("FINETUNE_CONFIG", REPO_ROOT / "configs" / "scratch" / "stack_tact_v4.yaml"))
cfg = Config.from_file(RUN_CONFIG)
print(cfg)

CHECKPOINT_DIR = Path(cfg.training.output_dir) / cfg.training.experiment_name
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

FileNotFoundError: [Errno 2] No such file or directory: '/rwthfs/rz/cluster/work/yn030245/worktrees/finetune-groot/finetune-groot/configs/scratch/stack_tact_v4.yaml'

## Dataset info and model adapter

`Gr00tAdapter.from_yaml` loads the edited dataset-info file. The table lists every modality the loader
provides; `use` marks the ones the model gets, in the order of the file. `index_map` states where each group
sits inside the assembled state and action vectors, since `robotdataset` itself only returns TED leaves and
does not track positions inside an assembled tensor. The adapter validates the file against a fresh listing of
the loader and a sampled batch when the dataset is built below, so a stale file fails with a message naming
what changed.

In [ ]:
from gr00t.data.embodiment_tags import EmbodimentTag
from robotdataset.adapters.gr00t import Gr00tAdapter
from robotdataset.info import parse_episodes

EMBODIMENT_TAG = EmbodimentTag.resolve(cfg.model.embodiment_tag)
TRAIN_EPISODES = parse_episodes(cfg.data.train_episodes)
VAL_EPISODES = parse_episodes(cfg.data.val_episodes)
assert TRAIN_EPISODES and VAL_EPISODES, "data.train_episodes and data.val_episodes must not be empty"
assert not set(TRAIN_EPISODES) & set(VAL_EPISODES), "train and validation episodes overlap"

adapter = Gr00tAdapter.from_yaml(REPO_ROOT / cfg.data.modality_file)
print(adapter.info.table())
modality_configs = adapter.modality_configs()
for role, entries in adapter.index_map().items():
    print(f"{role} vector:", [(e["group"], tuple(e["assembled"])) for e in entries])
modality_configs

## Build the model

`Gr00tN1d7Config` defaults + the tuning flags and precision from the `model:` group of the run file
(calibration-validated: projector + diffusion head trainable, backbone frozen, bf16).

In [ ]:
from gr00t.configs.model.gr00t_n1d7 import Gr00tN1d7Config
from gr00t.model.gr00t_n1d7.gr00t_n1d7 import Gr00tN1d7
from gr00t.model.gr00t_n1d7.processing_gr00t_n1d7 import Gr00tN1d7Processor

model_config = Gr00tN1d7Config(
    tune_llm=cfg.model.tune_llm,
    tune_visual=cfg.model.tune_visual,
    tune_projector=cfg.model.tune_projector,
    tune_diffusion_model=cfg.model.tune_diffusion_model,
)

model, loading_info = Gr00tN1d7.from_pretrained(
    cfg.model.base_model_path,
    tune_llm=model_config.tune_llm,
    tune_visual=model_config.tune_visual,
    tune_projector=model_config.tune_projector,
    tune_diffusion_model=model_config.tune_diffusion_model,
    tune_vlln=model_config.tune_vlln,
    state_dropout_prob=model_config.state_dropout_prob,
    backbone_trainable_params_fp32=model_config.backbone_trainable_params_fp32,
    load_bf16=cfg.model.load_bf16,
    output_loading_info=True,
    trust_remote_code=True,
)
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.1f}%)")

## Build the processor

The processor is created without statistics; the adapter sets them once the training dataset exists (see
"Normalisation statistics" below), because they are computed over the episodes that dataset loaded.
`action_horizon` of the model config must cover the adapter's action window, which the next cell checks.

In [ ]:
processor = Gr00tN1d7Processor(
    modality_configs={EMBODIMENT_TAG.value: modality_configs},  # parse_modality_configs expects {tag: {modality: cfg}}
    use_percentiles=model_config.use_percentiles,
    statistics=None,
    embodiment_id_mapping=None,
    image_crop_size=model_config.image_crop_size,
    image_target_size=model_config.image_target_size,
    random_rotation_angle=model_config.random_rotation_angle,
    color_jitter_params=model_config.color_jitter_params,
    model_name=model_config.model_name,
    model_type=model_config.backbone_model_type,
    formalize_language=model_config.formalize_language,
    max_state_dim=model_config.max_state_dim,
    max_action_dim=model_config.max_action_dim,
    apply_sincos_state_encoding=model_config.apply_sincos_state_encoding,
    max_action_horizon=model_config.action_horizon,
    use_albumentations=model_config.use_albumentations_transforms,
    extra_augmentation_config=model_config.extra_augmentation_config,
    shortest_image_edge=model_config.shortest_image_edge,
    crop_fraction=model_config.crop_fraction,
    use_relative_action=model_config.use_relative_action,
    exclude_state=model_config.exclude_state,
    state_dropout_prob=model_config.state_dropout_prob,
    use_mean_std=model_config.use_mean_std,
)
assert adapter.action_horizon <= model_config.action_horizon, (
    f"the modality file asks for an action horizon of {adapter.action_horizon}, "
    f"the model supports {model_config.action_horizon}"
)

## Load the data

The adapter builds the loader named in the modality file (`LeRobotVideoDataset`, `OXEDataset`, ...) with the
sampling windows the selection needs and checks the result against the file. `data.train_episodes` train;
`data.val_episodes` are held out (a fixed split recorded in the run file, not randomised per run).

In [ ]:
BATCH_SIZE = cfg.training.global_batch_size
train_ds = adapter.build_dataset(episodes=TRAIN_EPISODES, batch_size=BATCH_SIZE)
print("train episodes:", train_ds.num_episodes, "total steps:", len(train_ds))
val_ds = adapter.build_dataset(episodes=VAL_EPISODES, batch_size=BATCH_SIZE)
print("val episodes:", val_ds.num_episodes, "total steps:", len(val_ds))

## Normalisation statistics

`adapter.configure_processor` computes every statistic with the training loader (`generate_stats` per selected
leaf, cut into the groups of the selection, plus `generate_relative_action_stats` for `RELATIVE` action groups)
and hands them to `processor.set_statistics`. The held-out episodes do not take part, and nothing is read from or
written to the dataset's own metadata. Exact quantiles matter: `use_percentiles` defaults to `True`, so the
processor normalises with the 1st/99th percentiles.

In [ ]:
statistics = adapter.configure_processor(processor, train_ds, EMBODIMENT_TAG)
print(f"statistics from {train_ds.num_episodes} train episodes; processor has:", list(processor.statistics))

## Adapter: robotdataset batch -> `VLAStepData` -> processor -> model inputs

`adapter.to_model_batch` cuts the sampled TED leaves into the selected groups (images to HWC uint8, vectors to
`(T, D)`, actions to `(horizon, D)`), wraps them as `VLAStepData` (language from the selected text modality or the
file's default), runs each through the processor and collates. Smoke-test it on one training and one validation
batch before training.

In [ ]:
for _name, _ds in (("train", train_ds), ("val", val_ds)):
    _model_batch = adapter.to_model_batch(_ds.sample(), processor, EMBODIMENT_TAG)
    print(_name, "model batch keys:", list(_model_batch.keys()) if hasattr(_model_batch, "keys") else type(_model_batch))

## Training loop

Minimal: AdamW over trainable params only, cosine schedule, wandb logging, periodic checkpoint. No HF
`Trainer`: every step is explicit. Each checkpoint also gets the modality file the run used
(`dataset_info.yaml`) and a copy of the run file, so a checkpoint records which groups sit where in its state
and action vectors.

In [ ]:
MAX_STEPS = cfg.training.max_steps
WARMUP_STEPS = max(1, int(cfg.training.warmup_ratio * MAX_STEPS))

trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=cfg.training.learning_rate)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda step: min(1.0, (step + 1) / WARMUP_STEPS)
    * max(0.0, (MAX_STEPS - step) / max(1, MAX_STEPS - WARMUP_STEPS)),
)

wandb.init(
    project=cfg.training.wandb_project,
    name=cfg.training.experiment_name,
    mode="online" if cfg.training.use_wandb else "disabled",
    config={**cfg.to_dict(), "action_horizon": adapter.action_horizon, "trainable_params": trainable_params},
)

adapter.info.to_yaml(str(CHECKPOINT_DIR / "dataset_info.yaml"))
shutil.copy(RUN_CONFIG, CHECKPOINT_DIR / "run_config.yaml")

model.train()
processor.train()
losses = []
for step in range(MAX_STEPS):
    model_batch = adapter.to_model_batch(train_ds.sample(), processor, EMBODIMENT_TAG)
    model_batch = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in model_batch.items()}

    # bf16 backbone + fp32 layer norms: run the forward under autocast (what HF Trainer does with bf16=True)
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEVICE == "cuda"):
        out = model(**model_batch)
    loss = out.loss if hasattr(out, "loss") else out["loss"]

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(trainable, cfg.training.max_grad_norm)
    optimizer.step()
    scheduler.step()

    losses.append(loss.item())
    if step % cfg.training.logging_steps == 0 or step == MAX_STEPS - 1:
        wandb.log({"train/loss": loss.item(), "train/lr": scheduler.get_last_lr()[0]}, step=step)
        print(f"step {step:4d}  loss {loss.item():.4f}")
    if (step + 1) % cfg.training.save_steps == 0:
        ckpt_dir = CHECKPOINT_DIR / f"step_{step+1}"
        model.save_pretrained(ckpt_dir)
        processor.save_pretrained(ckpt_dir)
        print("saved checkpoint:", ckpt_dir)

wandb.finish()

## Loss curve

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(losses)
ax.set_xlabel("step")
ax.set_ylabel("training loss")
ax.set_title(f"{cfg.training.experiment_name}: training loss")
fig.tight_layout()
fig.savefig(CHECKPOINT_DIR / "loss.png", dpi=150)
plt.show()